In [1]:
# EDA uses the PostgreSQL warehouse because the raw CSV
# has already served its purpose in notebook 01.

from pathlib import Path
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Keep figures inside the notebook when cells are executed non-interactively.
%matplotlib inline

from sqlalchemy import create_engine
from dotenv import load_dotenv


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


load_dotenv(
    PROJECT_ROOT / ".env"
)


engine = create_engine(
    f"postgresql+psycopg2://"
    f"{os.getenv('DB_USER', 'ahadke')}:"
    f"{os.getenv('DB_PASSWORD', '')}@"
    f"{os.getenv('DB_HOST', 'localhost')}:"
    f"{os.getenv('DB_PORT', '5432')}/"
    f"{os.getenv('DB_NAME', 'amex_credit_risk')}"
)


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

In [2]:
# dim_customer should contain exactly one row per customer,
# making it the correct table for customer-level default analysis.

customers = pd.read_sql(
    """
    SELECT *
    FROM dim_customer
    """,
    engine,
)


print(
    "Customers:",
    f"{len(customers):,}",
)

print(
    "Unique IDs:",
    f"{customers['customer_ID'].nunique():,}",
)

print(
    "Default rate:",
    f"{customers['target'].mean():.4%}",
)


customers.head()

Customers: 458,913
Unique IDs: 458,913
Default rate: 25.8934%


,customer_ID,target,first_statement_date,last_statement_date,n_statements
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,0,2017-03-09,2018-03-13,13
1,00000fd6641609c6ece5454664794f0340ad84dddce9a2...,0,2017-03-01,2018-03-25,13
2,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,0,2017-03-11,2018-03-12,13
3,000041bdba6ecadd89a52d11886e8eaaec9325906c9723...,0,2017-03-31,2018-03-29,13
4,00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8a...,0,2017-03-30,2018-03-30,13


In [3]:
# Statement-count distribution shows how much
# longitudinal history is available per customer.

statement_distribution = (
    customers[
        "n_statements"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "n_statements"
    )
    .reset_index(
        name="customers"
    )
)


statement_distribution

,n_statements,customers
0,1,5120
1,2,6098
2,3,5778
3,4,4673
4,5,4671
5,6,5515
6,7,5198
7,8,6110
8,9,6411
9,10,6721


In [4]:
plt.figure(
    figsize=(9, 5)
)

plt.bar(
    statement_distribution[
        "n_statements"
    ],
    statement_distribution[
        "customers"
    ],
)

plt.xlabel(
    "Statements per customer"
)

plt.ylabel(
    "Number of customers"
)

plt.title(
    "Distribution of Customer Statement Histories"
)

plt.tight_layout()
plt.show()

In [5]:
# This checks whether customers with shorter histories
# have systematically different future-default rates.

default_by_history = (
    customers
    .groupby(
        "n_statements",
        as_index=False,
    )
    .agg(
        customers=(
            "customer_ID",
            "count",
        ),
        default_rate=(
            "target",
            "mean",
        ),
    )
)


default_by_history

,n_statements,customers,default_rate
0,1,5120,0.335742
1,2,6098,0.318465
2,3,5778,0.358602
3,4,4673,0.416221
4,5,4671,0.392635
5,6,5515,0.387670
6,7,5198,0.418430
7,8,6110,0.447300
8,9,6411,0.450164
9,10,6721,0.462282


In [6]:
plt.figure(
    figsize=(9, 5)
)

plt.plot(
    default_by_history[
        "n_statements"
    ],
    default_by_history[
        "default_rate"
    ],
    marker="o",
)

plt.xlabel(
    "Statements per customer"
)

plt.ylabel(
    "Default rate"
)

plt.title(
    "Default Rate by Statement History Length"
)

plt.tight_layout()
plt.show()

In [7]:
# We select several representative features spanning
# payment, balance, delinquency, risk and spend behavior.

EDA_FEATURES = [
    "P_2",
    "B_1",
    "B_2",
    "B_9",
    "D_39",
    "D_41",
    "D_44",
    "D_45",
    "D_47",
    "R_1",
    "R_2",
    "R_3",
    "S_3",
]


columns_sql = ", ".join(
    [
        f'f."{column}"'
        for column
        in EDA_FEATURES
    ]
)


query = f"""
SELECT
    f."customer_ID",
    f.statement_date,
    {columns_sql},
    d.target

FROM fact_statements f

JOIN dim_customer d
  ON f."customer_ID" = d."customer_ID"
"""


statements = pd.read_sql(
    query,
    engine,
)


statements[
    "statement_date"
] = pd.to_datetime(
    statements[
        "statement_date"
    ]
)


print(
    f"Rows pulled: {len(statements):,}"
)

Rows pulled: 5,531,451


In [8]:
# Using every statement in customer-level comparisons would give
# customers with more statements more influence.
# We therefore use each customer's latest observation.

latest = (
    statements
    .sort_values(
        [
            "customer_ID",
            "statement_date",
        ]
    )
    .groupby(
        "customer_ID",
        as_index=False,
    )
    .tail(1)
    .reset_index(drop=True)
)


print(
    "Latest-statement rows:",
    f"{len(latest):,}",
)

Latest-statement rows: 458,913


In [9]:
# Summary statistics provide a numeric comparison
# before looking at distribution plots.

summary_rows = []


for feature in EDA_FEATURES:

    for target in [
        0,
        1,
    ]:

        values = latest.loc[
            latest[
                "target"
            ] == target,
            feature,
        ]

        summary_rows.append(
            {
                "feature":
                    feature,

                "target":
                    target,

                "count":
                    values.notna().sum(),

                "mean":
                    values.mean(),

                "median":
                    values.median(),

                "std":
                    values.std(),

                "missing_pct":
                    values.isna().mean()
                    * 100,
            }
        )


distribution_summary = pd.DataFrame(
    summary_rows
)


distribution_summary

,feature,target,count,mean,median,std,missing_pct
0,P_2,0,337611,0.740946,0.784433,0.194934,0.727465
1,P_2,1,118333,0.333691,0.351809,0.211145,0.416568
2,B_1,0,340085,0.077590,0.019613,0.158374,0.000000
3,B_1,1,118828,0.314862,0.207850,0.305041,0.000000
4,B_2,0,340061,0.727481,0.817258,0.349741,0.007057
5,B_2,1,118821,0.202765,0.041610,0.319222,0.005891
6,B_9,0,340085,0.109942,0.009342,0.210127,0.000000
7,B_9,1,118828,0.480310,0.586549,0.346216,0.000000
8,D_39,0,340085,0.120790,0.008301,0.209228,0.000000
9,D_39,1,118828,0.432451,0.093782,0.653706,0.000000


In [10]:
# KDE curves show whether default and non-default customers
# occupy meaningfully different regions of each feature.

for feature in EDA_FEATURES:

    non_default = (
        latest.loc[
            latest[
                "target"
            ] == 0,
            feature,
        ]
        .dropna()
    )

    default = (
        latest.loc[
            latest[
                "target"
            ] == 1,
            feature,
        ]
        .dropna()
    )

    if (
        non_default.nunique() < 2
        or default.nunique() < 2
    ):
        continue

    plt.figure(
        figsize=(8, 5)
    )

    non_default.plot.kde(
        label="Non-default"
    )

    default.plot.kde(
        label="Default"
    )

    plt.xlabel(
        feature
    )

    plt.ylabel(
        "Density"
    )

    plt.title(
        f"{feature}: Default vs Non-default"
    )

    plt.legend()

    plt.tight_layout()
    plt.show()

In [11]:
# Missingness can also vary by outcome,
# so we compare it directly between target groups.

missing_rows = []


for feature in EDA_FEATURES:

    missing_rows.append(
        {
            "feature":
                feature,

            "non_default_missing_pct":
                latest.loc[
                    latest[
                        "target"
                    ] == 0,
                    feature,
                ]
                .isna()
                .mean()
                * 100,

            "default_missing_pct":
                latest.loc[
                    latest[
                        "target"
                    ] == 1,
                    feature,
                ]
                .isna()
                .mean()
                * 100,
        }
    )


missing_by_target = pd.DataFrame(
    missing_rows
)


missing_by_target

,feature,non_default_missing_pct,default_missing_pct
0,P_2,0.727465,0.416568
1,B_1,0.000000,0.000000
2,B_2,0.007057,0.005891
3,B_9,0.000000,0.000000
4,D_39,0.000000,0.000000
5,D_41,0.007057,0.005891
6,D_44,4.968758,4.541859
7,D_45,0.007057,0.005891
8,D_47,0.000000,0.000000
9,R_1,0.000000,0.000000


In [12]:
# Spearman correlation is useful here because many anonymized features
# are not normally distributed.

correlation_matrix = (
    latest[
        EDA_FEATURES
    ]
    .corr(
        method="spearman"
    )
)


correlation_matrix

,P_2,B_1,B_2,B_9,D_39,D_41,D_44,D_45,D_47,R_1,R_2,R_3,S_3
P_2,1.000000,-0.525054,0.546282,-0.528061,-0.243445,-0.255275,-0.616994,0.411122,0.452985,-0.311193,-0.187635,-0.291316,-0.452250
B_1,-0.525054,1.000000,-0.584858,0.738066,0.470309,0.211102,0.446221,-0.179311,-0.191759,0.231818,0.113663,0.128967,0.241790
B_2,0.546282,-0.584858,1.000000,-0.509105,-0.200360,-0.226978,-0.459546,0.212080,0.215155,-0.240080,-0.135048,-0.137094,-0.370478
B_9,-0.528061,0.738066,-0.509105,1.000000,0.340299,0.175515,0.433959,-0.327328,-0.294917,0.233770,0.115920,0.169931,0.249829
D_39,-0.243445,0.470309,-0.200360,0.340299,1.000000,0.205570,0.206802,-0.012858,-0.038316,0.143608,0.110414,0.047083,0.043616
D_41,-0.255275,0.211102,-0.226978,0.175515,0.205570,1.000000,0.176922,-0.041043,-0.063884,0.151038,0.129922,0.044902,0.117322
D_44,-0.616994,0.446221,-0.459546,0.433959,0.206802,0.176922,1.000000,-0.217111,-0.210420,0.272735,0.133506,0.184032,0.317617
D_45,0.411122,-0.179311,0.212080,-0.327328,-0.012858,-0.041043,-0.217111,1.000000,0.571914,-0.106760,-0.056978,-0.309617,-0.308680
D_47,0.452985,-0.191759,0.215155,-0.294917,-0.038316,-0.063884,-0.210420,0.571914,1.000000,-0.101131,-0.055673,-0.224834,-0.318087
R_1,-0.311193,0.231818,-0.240080,0.233770,0.143608,0.151038,0.272735,-0.106760,-0.101131,1.000000,0.178935,0.096668,0.173695


In [ ]:
# Known AMEX categoricals were missing from the numeric EDA sample,
# which blocked chi-square tests in notebook 03.
# D_63 and D_64 are TEXT in fact_statements; the others are DOUBLE PRECISION.
# We pull latest statements for these columns only and do not overwrite
# customer_feature_sample.parquet.

CATEGORICAL_FEATURES = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68",
]


cat_columns_sql = ", ".join(
    [
        f'f."{column}"'
        for column in CATEGORICAL_FEATURES
    ]
)


cat_query = f"""
SELECT DISTINCT ON (f."customer_ID")
    f."customer_ID",
    {cat_columns_sql}

FROM fact_statements f

ORDER BY
    f."customer_ID",
    f.statement_date DESC
"""


categorical_latest = pd.read_sql(
    cat_query,
    engine,
)


latest_with_categoricals = latest.merge(
    categorical_latest,
    on="customer_ID",
    how="left",
)


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

latest_with_categoricals.to_parquet(
    PROCESSED_DIR / "customer_feature_sample_with_categoricals.parquet",
    index=False,
)


print(
    "Expanded sample rows:",
    f"{len(latest_with_categoricals):,}",
)

print(
    "Categorical columns present:",
    [
        column
        for column in CATEGORICAL_FEATURES
        if column in latest_with_categoricals.columns
    ],
)


In [13]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


customers.to_parquet(
    PROCESSED_DIR
    / "dim_customer.parquet",
    index=False,
)


latest.to_parquet(
    PROCESSED_DIR
    / "customer_feature_sample.parquet",
    index=False,
)


distribution_summary.to_csv(
    PROCESSED_DIR
    / "eda_distribution_summary.csv",
    index=False,
)


print(
    "EDA outputs saved."
)

EDA outputs saved.


## Summary

This notebook is customer-level EDA on the PostgreSQL warehouse (`dim_customer` and `fact_statements`).

**What we did**
- Pulled one row per customer from `dim_customer` and confirmed uniqueness.
- Plotted statement-count distribution and default rate by history length.
- Pulled the full selected-feature history (**5,531,451** rows; no statement-row `LIMIT`).
- Kept each customer's **latest** statement so customers with longer histories are not over-weighted.
- Compared default vs non-default distributions, missingness by target, and Spearman correlations.
- Saved `dim_customer.parquet`, `customer_feature_sample.parquet`, and `eda_distribution_summary.csv`.
- Also saved `customer_feature_sample_with_categoricals.parquet` (same latest-statement grain plus 11 categorical columns). The numeric-only parquet was not overwritten.

**Results**
- **458,913** customers, **458,913** unique IDs, default rate **25.8934%**.
- **386,034** customers have 13 statements (default rate 23.2%). Shorter histories have higher default rates (about 32–46%).
- Latest-statement medians differ by target. Examples: **P_2** 0.784 (non-default) vs 0.352 (default); **B_2** 0.817 vs 0.042; **B_9** 0.009 vs 0.587.
- Missingness is similar by target for most features except **S_3** (22.3% non-default vs 7.6% default).
- Strongest Spearman pairs among the 13 features include **B_1–B_9** (0.74) and **P_2–D_44** (−0.62).

The numeric latest-statement table remains the input for the original Mann–Whitney tests. Categorical chi-square tests use the expanded parquet.
